In [1]:
import strawberryfields as sf
from strawberryfields.ops import *
import numpy as np
from scipy.optimize import minimize

In [2]:
# 1. Definição do Sistema
n_modes = 2  # Exemplo com 2 sítios (modos)
eng = sf.Engine(backend="gaussian") # Usamos backend gaussiano para a base

In [3]:
def circuit(params):
    """
    Constrói o ansatz fotônico: Squeezing -> Interferômetro -> Subtração
    params: [r1, r2, theta]
    """
    prog = sf.Program(n_modes)
    with prog.context as q:
        # Squeezing (Parâmetros r)
        Sgate(params[0]) | q[0] # squeeze trocar para rotação ou displaciment
        Sgate(params[1]) | q[1]
        
        # Interferômetro (Óptica Passiva - Parâmetro theta)
        BSgate(params[2], 0) | (q[0], q[1])
        
        # Nota: A subtração de fótons (não-gaussiana) para o cálculo do 
        # valor esperado no artigo é tratada analiticamente via matriz de covariância.
    
    state = eng.run(prog).state
    return state

In [4]:
def cost_function(params, target_u):
    """
    Calcula <H> para o modelo de Bose-Hubbard.
    H = -t * sum(ai* aj) + (U/2) * sum(ni(ni-1))
    """
    state = circuit(params)
    
    # Extração da Matriz de Covariância e Vetor de Médias
    cov = state.cov()
    means = state.means()
    
    # No artigo, eles usam o Teorema de Wick para calcular <H> 
    # a partir da matriz de covariância sem truncar o espaço de Fock.
    # Aqui simulamos uma estimativa simplificada da energia
    energy = estimate_bose_hubbard_energy(cov, means, target_u)
    
    return energy

In [5]:
def estimate_bose_hubbard_energy(cov, means, U):
    # Simplificação: Em um cenário real, aqui entraria a expansão 
    # de Wick para o estado não-gaussiano (após subtração de fótons).
    # Este valor representa a "energia" que o otimizador tenta minimizar.
    n1 = (cov[0,0] + cov[n_modes, n_modes] - 1) / 2
    n2 = (cov[1,1] + cov[n_modes+1, n_modes+1] - 1) / 2
    interaction = (U / 2) * (n1**2 + n2**2) 
    return n1 + n2 + interaction # Exemplo de termo de energia


In [6]:
# 2. Execução da Otimização Clássica (L-BFGS-B como no artigo)
initial_params = np.array([0.5, 0.5, np.pi/4])
res = minimize(cost_function, initial_params, args=(0.1,), method='L-BFGS-B')

In [7]:
print(f"Parâmetros otimizados: {res.x}")
print(f"Energia do Estado Fundamental estimada: {res.fun}")

Parâmetros otimizados: [0.5        0.5        0.78539816]
Energia do Estado Fundamental estimada: 3300898598502.402
